# Data acquisition and validation

Pulls price and statement data once, saves it to `data/raw/`, and checks it against the deal facts before anything downstream touches it.

In [1]:

import sys, warnings
sys.path.insert(0, r"/Users/shaan/Desktop/FAM/sunpharma-organon-merger-arbitrage")
warnings.filterwarnings("ignore")

from src import config as cfg, data, checks
import pandas as pd

print("tickers:", cfg.ALL_TICKERS)
print("history start:", cfg.HISTORY_START)
print("snapshot date:", cfg.SNAPSHOT_DATE)


tickers: ['OGN', 'SUNPHARMA.NS', '^GSPC', '^NSEI', '^VIX', 'INR=X', '^TNX', 'MRK', 'PFE', 'VTRS', 'TEVA', 'CIPLA.NS', 'DRREDDY.NS', 'LUPIN.NS']
history start: 2021-01-01
snapshot date: 2026-08-06


## Price history

In [2]:

prices = data.pull_all_prices()
data.save_prices(prices)

rows = []
for t, df in prices.items():
    rows.append({
        "ticker": t,
        "n_obs": len(df),
        "start": df.index.min().date(),
        "end": df.index.max().date(),
        "nulls": int(df["Close"].isna().sum()),
        "last_close": round(df["Close"].iloc[-1], 3),
    })
summary = pd.DataFrame(rows).set_index("ticker")
summary


,n_obs,start,end,nulls,last_close
ticker,,,,,
OGN,1312,2021-05-14,2026-08-05,0,13.570
SUNPHARMA.NS,1386,2021-01-01,2026-08-06,0,1951.000
^GSPC,1403,2021-01-04,2026-08-05,0,7723.550
^NSEI,1382,2021-01-01,2026-08-06,0,24636.000
^VIX,1405,2021-01-04,2026-08-06,0,16.000
INR=X,1455,2021-01-01,2026-08-06,0,95.210
^TNX,1404,2021-01-04,2026-08-06,0,4.643
MRK,1403,2021-01-04,2026-08-05,0,128.330
PFE,1403,2021-01-04,2026-08-05,0,25.810


In [3]:

assert summary["nulls"].sum() == 0, "unexpected nulls in price history"
assert (summary["n_obs"] > 1000).all(), "unexpectedly short history for one or more tickers"
print("price integrity checks passed")


price integrity checks passed


## Financial statements

In [4]:

statement_tickers = [cfg.TARGET, cfg.ACQUIRER] + cfg.US_PEERS + cfg.IN_PEERS
for ticker in statement_tickers:
    stmts = data.pull_statements(ticker)
    data.save_statements(ticker, stmts)
    inc = stmts["income_stmt"]
    print(ticker, "income statement:", inc.shape, [c.year for c in inc.columns])


OGN income statement: (44, 5) [2025, 2024, 2023, 2022, 2021]


SUNPHARMA.NS income statement: (53, 4) [2026, 2025, 2024, 2023]


MRK income statement: (45, 5) [2025, 2024, 2023, 2022, 2021]


PFE income statement: (55, 5) [2025, 2024, 2023, 2022, 2021]


VTRS income statement: (43, 4) [2025, 2024, 2023, 2022]


TEVA income statement: (52, 4) [2025, 2024, 2023, 2022]


CIPLA.NS income statement: (54, 4) [2026, 2025, 2024, 2023]


DRREDDY.NS income statement: (54, 5) [2026, 2025, 2024, 2023, 2022]


LUPIN.NS income statement: (53, 4) [2026, 2025, 2024, 2023]


## Reconciliation against the deal terms

The 16-Jan-2026 and 9-Apr-2026 closes should reproduce the reported 60.4% and 102.9% premia to the $14.00 offer. Enterprise value reconciles against the FY2025 (2025-12-31) balance sheet, the last annual filing before the deal was signed — not a later, post-announcement quarter.

In [5]:

ogn = prices[cfg.TARGET]["Close"]

price_16jan = ogn.loc[ogn.index <= pd.Timestamp(cfg.DEAL["leak_date"])].iloc[-1]
price_9apr = ogn.loc[ogn.index <= pd.Timestamp(cfg.DEAL["unaffected_date"])].iloc[-1]
price_today = ogn.iloc[-1]

print(f"16-Jan-2026 close : ${price_16jan:.2f}")
print(f"9-Apr-2026 close  : ${price_9apr:.2f}")
print(f"current close     : ${price_today:.2f}")


16-Jan-2026 close : $8.76
9-Apr-2026 close  : $6.91
current close     : $13.57


In [6]:

ogn_stmts = data.load_statements(cfg.TARGET)
bs = ogn_stmts["balance_sheet"]

fy2025_col = [c for c in bs.columns if c.year == 2025][0]
debt_fy2025 = bs.loc["Total Debt", fy2025_col]
cash_fy2025 = bs.loc["Cash And Cash Equivalents", fy2025_col]
net_debt_fy2025 = debt_fy2025 - cash_fy2025

print(f"FY2025 balance sheet date : {fy2025_col.date()}")
print(f"FY2025 total debt         : ${debt_fy2025/1e9:.3f}B")
print(f"FY2025 cash               : ${cash_fy2025/1e6:.1f}M")
print(f"FY2025 net debt           : ${net_debt_fy2025/1e9:.3f}B")


FY2025 balance sheet date : 2025-12-31
FY2025 total debt         : $8.644B
FY2025 cash               : $574.0M
FY2025 net debt           : $8.070B


In [7]:

results = checks.run_all_price_checks(
    price_16jan=price_16jan,
    price_9apr=price_9apr,
    shares_outstanding=cfg.DEAL["shares_outstanding"],
    net_debt_usd=net_debt_fy2025,
)


RECONCILIATION SUITE
  16-Jan premium   : 59.8%  (reported 60.4%)  PASS
  9-Apr premium    : 102.6%  (reported 102.9%) PASS
  Implied EV       : $11.747B  (reported $11.75B, diff -0.03%) PASS


## Snapshot reference table

Saved to `data/final/` for reuse in later notebooks.

In [8]:

snapshot = pd.Series({
    "ogn_price_16jan": price_16jan,
    "ogn_price_9apr": price_9apr,
    "ogn_price_current": price_today,
    "ogn_net_debt_fy2025_usd": net_debt_fy2025,
    "ogn_shares_outstanding": cfg.DEAL["shares_outstanding"],
    "offer_price": cfg.DEAL["offer_price_usd"],
    "implied_equity_value_usd": cfg.DEAL["shares_outstanding"] * cfg.DEAL["offer_price_usd"],
    "implied_ev_usd": cfg.DEAL["shares_outstanding"] * cfg.DEAL["offer_price_usd"] + net_debt_fy2025,
})
snapshot.to_csv(cfg.DATA_FINAL / "deal_snapshot.csv", header=["value"])
snapshot


ogn_price_16jan             8.760000e+00
ogn_price_9apr              6.910000e+00
ogn_price_current           1.357000e+01
ogn_net_debt_fy2025_usd     8.070000e+09
ogn_shares_outstanding      2.626094e+08
offer_price                 1.400000e+01
implied_equity_value_usd    3.676532e+09
implied_ev_usd              1.174653e+10
dtype: float64